# 02_teacher — BirdNET Teacher Logits

**Run once, in the `PocketBirdNET Teacher (.venv-birdnet)` kernel.**

This notebook produces **`data/teacher_logits.npy`** — shape `(N_train, 11)`,
one row per row of `train.npz`, row-aligned.  Nothing else is written or modified.

## What this notebook does
1. Verifies prerequisites (`train_window_map.csv`, `train.npz`, `manifest.csv`).
2. Loads BirdNET via `birdnetlib` and maps its 6522-label output to our 11 classes.
3. Re-downloads each training recording from Xeno-Canto, extracts the same
   3-second windows notebook 01 used (matched by time position), runs BirdNET,
   then **deletes the mp3** immediately (process-and-delete).
4. Converts BirdNET's multi-label sigmoid outputs to pseudo-logits (inverse-sigmoid)
   so notebook 03 can apply temperature scaling `T=4` itself.
5. Propagates each unique window's logit vector to every `train.npz` row that
   derives from it (originals + augmented copies).
6. Runs sanity checks and prints a teacher–ground-truth agreement rate.

## Key design decisions (documented for reproducibility)

**Window timing** — matched to notebook 01 exactly: at 48 kHz (BirdNET's native
rate) window `k` starts at sample `k × 72000` (hop = 1.5 s × 48000 Hz).

**Background class** — BirdNET has no background class.  We define:
```
bg_confidence = max(0, 1 − max(conf_0..conf_9))
```
Rationale: if BirdNET is highly confident about any of our 10 species, the window
is not background; if BirdNET is not confident about any of them, the window is
background.  This is a soft, monotone rule with no free threshold parameter.

**Stored representation** — pseudo-logits via the inverse sigmoid:
```
logit(p) = log(p / (1 − p + ε))
```
We store pre-softmax logits, **not** a softmaxed distribution, so notebook 03
can apply any temperature `T` without re-running this notebook.

**Alignment contract** — `teacher_logits[i]` corresponds to `train.npz X[i]`.
The mapping is `train_window_map.csv`, written by the updated §7 of notebook 01.
If that file is absent, this notebook raises an error before downloading anything.

**Env note** — This notebook uses `birdnetlib` / `birdnet-analyzer` (`.venv-birdnet`
kernel). It does **not** import TensorFlow from the main env.

## §1  Config & imports

In [1]:
import os, sys, json, time, pathlib, warnings, logging
import numpy as np
import pandas as pd
import requests
import librosa
import soundfile as sf
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("02_teacher")

# ── Repo root ─────────────────────────────────────────────────────────────────
try:
    _here = pathlib.Path(__file__).resolve().parent
except NameError:
    _here = pathlib.Path.cwd()

REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
DATA_DIR  = REPO_ROOT / "data"

# ── API key ───────────────────────────────────────────────────────────────────
XC_API_KEY = os.environ.get("XENO_CANTO_API_KEY", "")
if not XC_API_KEY:
    raise EnvironmentError(
        "Missing XENO_CANTO_API_KEY.  Run:\n"
        "  export XENO_CANTO_API_KEY=<key>\n"
        "then restart the kernel."
    )

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_NPZ_F    = DATA_DIR / "train.npz"
TRAIN_MAP_F    = DATA_DIR / "train_window_map.csv"
MANIFEST_F     = DATA_DIR / "manifest.csv"
TEACHER_NPY_F  = DATA_DIR / "teacher_logits.npy"
PROGRESS_F     = DATA_DIR / "_teacher_progress.json"
TMP_DIR        = DATA_DIR / "_tmp_audio"
TMP_DIR.mkdir(exist_ok=True)

# ── BirdNET native sample rate (48 kHz) ───────────────────────────────────────
BN_SR          = 48_000          # BirdNET processes audio at 48 kHz
BN_WIN_SAMPLES = BN_SR * 3       # 3-second window = 144 000 samples
# Notebook 01 used 50% overlap (hop = 1.5 s) at 16 kHz.
# Same hop in seconds → at 48 kHz: hop = 1.5 × 48 000 = 72 000 samples.
BN_HOP_SAMPLES = int(1.5 * BN_SR)   # 72 000

# ── Class list — must match notebook 01 SPECIES list exactly ──────────────────
SPECIES = [
    "American Robin",          # 0
    "Black-capped Chickadee",  # 1
    "Steller's Jay",           # 2
    "Northern Flicker",        # 3
    "Song Sparrow",            # 4
    "Anna's Hummingbird",      # 5
    "Dark-eyed Junco",         # 6
    "American Crow",           # 7
    "Pacific Wren",            # 8
    "House Finch",             # 9
    "background",              # 10
]
LABEL_MAP = {name: idx for idx, name in enumerate(SPECIES)}
N_CLASSES  = len(SPECIES)   # 11

# ── BirdNET label strings for our 10 species (format: "Scientific_Common") ────
# Verified against analyzer.labels at probe time.
BN_LABELS_FOR_SPECIES = {
    "American Robin":          "Turdus migratorius_American Robin",
    "Black-capped Chickadee":  "Poecile atricapillus_Black-capped Chickadee",
    "Steller's Jay":           "Cyanocitta stelleri_Steller's Jay",
    "Northern Flicker":        "Colaptes auratus_Northern Flicker",
    "Song Sparrow":            "Melospiza melodia_Song Sparrow",
    "Anna's Hummingbird":      "Calypte anna_Anna's Hummingbird",
    "Dark-eyed Junco":         "Junco hyemalis_Dark-eyed Junco",
    "American Crow":           "Corvus brachyrhynchos_American Crow",
    "Pacific Wren":            "Troglodytes pacificus_Pacific Wren",
    "House Finch":             "Haemorhous mexicanus_House Finch",
}

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"API key   : set (length={len(XC_API_KEY)})")

Repo root : /Users/rishabhgoenka/PocketBirdNET
Data dir  : /Users/rishabhgoenka/PocketBirdNET/data
API key   : set (length=40)


## §2  Prerequisite check

This cell raises immediately if notebook 01 has not been run correctly.

> **If `train_window_map.csv` is missing:** notebook 01's `build_split` was not
> updated to emit the source-window map.  Re-open notebook 01, re-run cells §6
> and §7 only (takes < 1 min; the `windows_raw.npz` cache is preserved), then
> return here.

In [2]:
# ── Check required input files ────────────────────────────────────────────────
missing = [f for f in [TRAIN_NPZ_F, TRAIN_MAP_F, MANIFEST_F] if not f.exists()]
if missing:
    raise FileNotFoundError(
        "Required files missing:\n" +
        "\n".join(f"  {f}" for f in missing) +
        ("\n\nFor train_window_map.csv: re-run notebook 01 §6 and §7 "
         "(cells that build split arrays and save .npz).  Takes < 1 min."
         if TRAIN_MAP_F in missing else "")
    )

# ── Load and validate the map ─────────────────────────────────────────────────
train_map = pd.read_csv(TRAIN_MAP_F, dtype=str, index_col="train_idx")
train_d   = np.load(TRAIN_NPZ_F)
N_TRAIN   = len(train_d["X"])

assert len(train_map) == N_TRAIN, (
    f"train_window_map has {len(train_map)} rows but train.npz has {N_TRAIN} rows. "
    "Re-run notebook 01 §6 and §7."
)
assert set(train_map.columns) >= {"recording_id", "window_idx"}, (
    f"Unexpected columns in train_window_map: {list(train_map.columns)}"
)

manifest  = pd.read_csv(MANIFEST_F, dtype=str)

# Unique (recording_id, window_idx) pairs we need BirdNET predictions for
unique_windows = (
    train_map[["recording_id", "window_idx"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_recs = unique_windows["recording_id"].unique()

print(f"train.npz rows          : {N_TRAIN}")
print(f"unique (rec, win) pairs : {len(unique_windows)}  (should equal raw train rows)")
print(f"unique recordings       : {len(unique_recs)}")
print()
print("Prerequisites: OK")

train.npz rows          : 6292
unique (rec, win) pairs : 3146  (should equal raw train rows)
unique recordings       : 230

Prerequisites: OK


## §3  Load BirdNET and build species-index map

This cell loads the BirdNET TFLite model via `birdnetlib`.  First run downloads
the model weights (~50 MB) automatically.  It also verifies that every one of our
10 species has an exact match in BirdNET's label list.

In [3]:
from birdnetlib.analyzer import Analyzer

analyzer = Analyzer()
bn_labels = analyzer.labels   # list of 6522 strings like "Turdus migratorius_American Robin"
bn_label_to_idx = {lbl: i for i, lbl in enumerate(bn_labels)}

print(f"BirdNET loaded.  Label count: {len(bn_labels)}")
print(f"Example labels: {bn_labels[:3]}")
print()

# Resolve each of our 10 species to a BirdNET label index
species_to_bn_idx = {}   # our species name → BirdNET label index
print("Species → BirdNET label mapping:")
for sp_name, bn_label in BN_LABELS_FOR_SPECIES.items():
    if bn_label not in bn_label_to_idx:
        # Fallback: search by scientific name substring
        sci = bn_label.split("_")[0]
        matches = [l for l in bn_labels if l.startswith(sci)]
        if not matches:
            raise KeyError(
                f"Cannot find BirdNET label for {sp_name!r}.\n"
                f"  Tried: {bn_label!r}\n"
                f"  Update BN_LABELS_FOR_SPECIES in §1."
            )
        bn_label = matches[0]
        print(f"  [fallback] {sp_name}: {bn_label}")
    species_to_bn_idx[sp_name] = bn_label_to_idx[bn_label]
    print(f"  {sp_name:30s} → [{species_to_bn_idx[sp_name]:4d}] {bn_label}")

# Ordered index array: bn_indices[k] = BirdNET index for SPECIES[k] (k=0..9)
# background (k=10) has no BirdNET index — derived from the other 10.
BN_INDICES = np.array([species_to_bn_idx[sp] for sp in SPECIES[:10]], dtype=np.int32)
print(f"\nBN_INDICES: {BN_INDICES}")

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
BirdNET loaded.  Label count: 6522
Example labels: ['Abroscopus albogularis_Rufous-faced Warbler', 'Abroscopus schisticeps_Black-faced Warbler', 'Abroscopus superciliaris_Yellow-bellied Warbler']

Species → BirdNET label mapping:
  American Robin                 → [6254] Turdus migratorius_American Robin
  Black-capped Chickadee         → [4771] Poecile atricapillus_Black-capped Chickadee
  Steller's Jay                  → [1749] Cyanocitta stelleri_Steller's Jay
  Northern Flicker               → [1455] Colaptes auratus_Northern Flicker
  Song Sparrow                   → [3514] Melospiza melodia_Song Sparrow
  Anna's Hummingbird             → [ 893] Calypte anna_Anna's Hummingbird
  Dark-eyed Junco                → [2992] Junco hyemalis_Dark-eyed Junco
  American Crow                  → [1575] Corvus brachyrhynchos_American Crow
  Pacific Wren                   → [6183] Troglodytes p

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## §4  Download → BirdNET inference → delete

For each unique training recording:
1. Download mp3 from Xeno-Canto to `data/_tmp_audio/`.
2. Decode at **48 kHz mono** (BirdNET's native rate).
3. For each unique `window_idx` needed from this recording, cut the 3-second
   window at the same **time position** as notebook 01 (hop = 1.5 s).
4. Call `analyzer.predict(chunk)` → 6522-dim sigmoid probability vector.
5. Extract the 10-species confidences; compute background confidence.
6. **Delete the mp3** immediately after processing all its windows.

**Resumable** — a progress file is saved after each recording.
Re-running this cell skips already-processed recordings.

Writes intermediate results to `data/_teacher_progress.json` so nothing is lost
if the cell is interrupted.

In [4]:
def download_mp3(file_url: str, dest: pathlib.Path, rid: str) -> bool:
    """Download one XC mp3. Returns True on success."""
    try:
        params = {"key": XC_API_KEY} if "xeno-canto.org" in file_url else {}
        r = requests.get(file_url, params=params, timeout=60, stream=True)
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(65536):
                fh.write(chunk)
        return True
    except Exception as exc:
        log.warning("Download failed for %s: %s", rid, exc)
        return False


def run_birdnet_on_window(
    audio_48k: np.ndarray,
    window_idx: int,
) -> np.ndarray:
    """Extract one 3-second window and run BirdNET on it.

    Parameters
    ----------
    audio_48k   : full recording at 48 kHz, float32 mono
    window_idx  : which 3-second / 50%-overlap window to use

    Returns
    -------
    np.ndarray  : shape (6522,) — BirdNET sigmoid probabilities for all labels
    """
    start = window_idx * BN_HOP_SAMPLES
    chunk = audio_48k[start : start + BN_WIN_SAMPLES].astype(np.float32)

    # Pad if recording ends before the window fills
    if len(chunk) < BN_WIN_SAMPLES:
        chunk = np.pad(chunk, (0, BN_WIN_SAMPLES - len(chunk)))

    # analyzer.predict returns shape (1, n_labels) after sigmoid
    pred = analyzer.predict(chunk)   # (1, 6522)
    return pred[0]                   # (6522,)


def bn_pred_to_confidence_11(pred_6522: np.ndarray) -> np.ndarray:
    """Convert a BirdNET (6522,) prediction to our 11-dim confidence vector.

    Background confidence = max(0, 1 − max(species_conf_0..9)).
    Returns raw sigmoid confidences (NOT logits) in [0, 1].
    """
    species_conf = pred_6522[BN_INDICES]   # (10,) — our 10 species
    bg_conf = max(0.0, 1.0 - float(species_conf.max()))
    return np.append(species_conf, bg_conf).astype(np.float32)  # (11,)


# ── Build lookup: recording_id → file_url (from manifest) ────────────────────
rec_url_map = dict(zip(manifest["recording_id"], manifest["file_url"]))

# ── Build lookup: recording_id → set of window_idx needed ────────────────────
rec_to_windows = (
    unique_windows
    .groupby("recording_id")["window_idx"]
    .apply(lambda s: sorted(s.astype(int).tolist()))
    .to_dict()
)

# ── Load progress ─────────────────────────────────────────────────────────────
# Stores {"rec_id": {"win_idx": [conf_0..conf_10], ...}, ...}
if PROGRESS_F.exists():
    cached = json.loads(PROGRESS_F.read_text())
    log.info("Resuming: %d recordings already processed.", len(cached))
else:
    cached = {}

todo_recs = [rid for rid in unique_recs if rid not in cached]
log.info("%d recordings to process.", len(todo_recs))

# ── Main loop ─────────────────────────────────────────────────────────────────
for rid in tqdm(todo_recs, desc="BirdNET inference"):
    url = rec_url_map.get(rid, "")
    if not url or url == "nan":
        log.warning("%s: no file URL — filling with uniform logits.", rid)
        cached[rid] = {
            str(w): [1.0 / N_CLASSES] * N_CLASSES
            for w in rec_to_windows.get(rid, [])
        }
        continue

    tmp = TMP_DIR / f"{rid}.mp3"
    audio_48k = None
    try:
        if not download_mp3(url, tmp, rid):
            raise IOError("download failed")

        audio_48k, _ = librosa.load(str(tmp), sr=BN_SR, mono=True)

        rec_results = {}
        for w_idx in rec_to_windows.get(rid, []):
            pred = run_birdnet_on_window(audio_48k, w_idx)
            conf11 = bn_pred_to_confidence_11(pred)
            rec_results[str(w_idx)] = conf11.tolist()

        cached[rid] = rec_results

    except Exception as exc:
        log.warning("%s: error (%s) — filling with uniform logits.", rid, exc)
        cached[rid] = {
            str(w): [1.0 / N_CLASSES] * N_CLASSES
            for w in rec_to_windows.get(rid, [])
        }
    finally:
        if tmp.exists():
            tmp.unlink()   # always delete — never retain raw audio

    # Save progress after each recording
    PROGRESS_F.write_text(json.dumps(cached))
    time.sleep(0.3)

log.info("BirdNET inference complete.  %d unique recordings processed.", len(cached))

2026-06-02 21:07:55,140 INFO Resuming: 68 recordings already processed.
2026-06-02 21:07:55,141 INFO 162 recordings to process.


BirdNET inference:   0%|          | 0/162 [00:00<?, ?it/s]

2026-06-02 21:25:59,362 INFO BirdNET inference complete.  230 unique recordings processed.


## §5  Convert confidences to pseudo-logits

BirdNET outputs sigmoid probabilities `p ∈ (0, 1)`.  We convert via inverse
sigmoid so that notebook 03 can apply temperature scaling:

```
logit(p) = log(p / (1 − p + ε))
```

We store **pre-softmax logits**, not a distribution.  Notebook 03 does:
```python
soft_targets = softmax(teacher_logits[i] / T)   # T=4 by default
```

In [5]:
EPS = 1e-7   # numerical stability for logit conversion

def sigmoid_to_logit(p: np.ndarray) -> np.ndarray:
    """Inverse sigmoid: logit(p) = log(p / (1 − p + ε))."""
    p = np.clip(p, EPS, 1.0 - EPS)
    return np.log(p / (1.0 - p)).astype(np.float32)


# Build window_key → logit vector lookup
# window_key = (recording_id, window_idx_str)
window_logit_cache: dict = {}   # (rid, str(widx)) → (11,) float32 logit array

for rid, win_dict in cached.items():
    for w_str, conf11_list in win_dict.items():
        conf11 = np.array(conf11_list, dtype=np.float32)
        window_logit_cache[(rid, w_str)] = sigmoid_to_logit(conf11)

print(f"Unique window logit vectors computed: {len(window_logit_cache)}")

# Spot-check a few entries
for (rid, widx), logits in list(window_logit_cache.items())[:3]:
    print(f"  ({rid}, win={widx}): logits={logits.round(2)}")

Unique window logit vectors computed: 3146
  (812729, win=0): logits=[-2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3]
  (812729, win=1): logits=[-2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3]
  (812729, win=2): logits=[-2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3 -2.3]


## §6  Assemble `teacher_logits.npy` aligned to `train.npz`

`train_window_map.csv` row `i` tells us the `(recording_id, window_idx)` that
`train.npz[i]` was derived from.  We look up the pre-computed logit vector for
that window — original rows and their augmented copies both inherit the same
teacher vector (augmentation is a spectrogram transform; BirdNET runs on raw audio,
so the teacher sees the unaugmented source).

In [6]:
# Guard: catch an empty cache before writing a silent all-zero file.
# This happens when §4 was skipped or kernel state was lost.
if len(window_logit_cache) == 0:
    raise RuntimeError(
        "window_logit_cache is empty — §5 produced no logit vectors.\\n"
        "Most likely cause: kernel was restarted after §4 ran, losing 'cached' "
        "from memory, and the _teacher_progress.json was already deleted.\\n"
        "Fix: re-run §4 (it will re-download and re-infer), then §5 and §6."
    )

teacher_logits = np.zeros((N_TRAIN, N_CLASSES), dtype=np.float32)
missing_keys = []

# iterrows() returns the DataFrame index as i — cast to int for numpy indexing.
for i, row in train_map.iterrows():
    row_idx = int(i)
    key = (str(row["recording_id"]), str(int(float(row["window_idx"]))))
    if key not in window_logit_cache:
        missing_keys.append((row_idx, key))
        teacher_logits[row_idx] = 0.0   # placeholder — flagged below
    else:
        teacher_logits[row_idx] = window_logit_cache[key]

if missing_keys:
    log.warning(
        "%d train rows had no logit in cache (first 5: %s).  "
        "These rows received all-zero logits — check download errors above.",
        len(missing_keys), missing_keys[:5],
    )

print(f"teacher_logits shape: {teacher_logits.shape}")
print(f"Missing keys: {len(missing_keys)} (should be 0)")

teacher_logits shape: (6292, 11)
Missing keys: 0 (should be 0)


## §7  Save `data/teacher_logits.npy`

In [7]:
# Sanity gate: refuse to save if the array is all-zero (indicates empty cache).
if teacher_logits.std() < 1e-6:
    raise ValueError(
        "teacher_logits is all-zero — aborting save to avoid overwriting good data.\\n"
        "Re-run §4 to rebuild the logit cache, then §5 and §6."
    )

np.save(TEACHER_NPY_F, teacher_logits)
size_mb = TEACHER_NPY_F.stat().st_size / 1e6
print(f"Saved {TEACHER_NPY_F}  ({size_mb:.2f} MB)")
print(f"Shape: {teacher_logits.shape}  dtype: {teacher_logits.dtype}")

# Keep the progress file until §8 confirms the data is good.
# It is deleted at the end of §8 after checks pass.
print("Progress file preserved until §8 sanity checks pass.")

Saved /Users/rishabhgoenka/PocketBirdNET/data/teacher_logits.npy  (0.28 MB)
Shape: (6292, 11)  dtype: float32
Progress file preserved until §8 sanity checks pass.


## §8  Sanity checks

These checks verify the teacher targets are usable for distillation.  Each check
prints PASS or a specific failure reason.

The **agreement rate** is not expected to be high — BirdNET was trained on clean
close-mic recordings and may struggle with the same Xeno-Canto recordings we used
(domain and quality variation).  But it should be **well above chance (9%)** on
clean bird windows.  If it is near random, the window-timing alignment is wrong.

In [8]:
# ── Reload from disk ──────────────────────────────────────────────────────────
tl = np.load(TEACHER_NPY_F)   # (N_train, 11)
y_train = np.load(TRAIN_NPZ_F)["y"]

print("=" * 62)
print("CHECK 1: Shape and dtype")
assert tl.shape == (N_TRAIN, N_CLASSES), f"Shape mismatch: {tl.shape}"
assert tl.dtype == np.float32
assert len(y_train) == N_TRAIN
print(f"  teacher_logits.shape = {tl.shape}  dtype = {tl.dtype}  PASS")

print()
print("CHECK 2: No NaN or Inf")
n_nan = int(np.isnan(tl).sum())
n_inf = int(np.isinf(tl).sum())
print(f"  NaN count: {n_nan}  Inf count: {n_inf}")
assert n_nan == 0 and n_inf == 0, "NaN/Inf found in teacher_logits!"
print("  PASS")

print()
print("CHECK 3: Row-alignment with train.npz")
assert tl.shape[0] == len(y_train), f"Row count mismatch: {tl.shape[0]} vs {len(y_train)}"
print(f"  teacher_logits rows = {tl.shape[0]}, train.npz rows = {len(y_train)}  PASS")

CHECK 1: Shape and dtype
  teacher_logits.shape = (6292, 11)  dtype = float32  PASS

CHECK 2: No NaN or Inf
  NaN count: 0  Inf count: 0
  PASS

CHECK 3: Row-alignment with train.npz
  teacher_logits rows = 6292, train.npz rows = 6292  PASS


In [9]:
print("=" * 62)
print("CHECK 4: Background windows put mass on background class (col 10)")

bg_label = LABEL_MAP["background"]   # = 10
bg_mask  = (y_train == bg_label)
n_bg     = bg_mask.sum()

if n_bg == 0:
    print("  No background windows in train set — skipping.")
else:
    bg_logits = tl[bg_mask]   # (N_bg, 11)
    # Softmax at T=1 to get a distribution
    def softmax(x):
        e = np.exp(x - x.max(axis=-1, keepdims=True))
        return e / e.sum(axis=-1, keepdims=True)
    bg_probs = softmax(bg_logits)   # (N_bg, 11)
    bg_argmax = bg_probs.argmax(axis=1)
    bg_class_frac = (bg_argmax == bg_label).mean()
    print(f"  Background windows: {n_bg}")
    print(f"  Fraction with argmax = background class: {bg_class_frac:.2%}")
    print(f"  Mean probability on background class:    {bg_probs[:, bg_label].mean():.3f}")
    if bg_class_frac < 0.3:
        print("  WARNING: low fraction — background teacher signal is weak (common if BirdNET")
        print("           detects species in ambient recordings).")
    else:
        print("  PASS")

CHECK 4: Background windows put mass on background class (col 10)
  Background windows: 546
  Fraction with argmax = background class: 100.00%
  Mean probability on background class:    1.000
  PASS


In [10]:
print("=" * 62)
print("CHECK 5: Teacher–ground-truth agreement on CLEAN (non-augmented) train windows")
print("         (augmented copies have the same teacher vector as their source,")
print("          so we isolate originals to avoid double-counting)")
print()

# Originals are the first N_raw rows in the pre-permutation array.
# We identify them via train_window_map: unique (rec, win) pairs appear exactly
# once in the map for originals and once for their augmented copy.  A clean
# proxy: rows where this is the FIRST occurrence of (recording_id, window_idx).
dup_mask = train_map.duplicated(subset=["recording_id", "window_idx"], keep="first")
orig_mask = ~dup_mask.values   # True for first-occurrence rows (originals)
n_orig = orig_mask.sum()

print(f"  Original (non-augmented) rows identified: {n_orig} / {N_TRAIN}")

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

tl_orig   = tl[orig_mask]
y_orig    = y_train[orig_mask]
prob_orig = softmax(tl_orig)           # T=1 for this diagnostic
pred_orig = prob_orig.argmax(axis=1)   # teacher argmax

# Overall agreement (teacher argmax == ground-truth label)
agreement_all = (pred_orig == y_orig).mean()
print(f"  Overall agreement (teacher argmax == ground truth): {agreement_all:.2%}")
print(f"  Chance baseline (11 classes):                       {1/N_CLASSES:.2%}")
print()

# Per-class breakdown (exclude background — BirdNET's signal there is indirect)
print("  Per-species agreement (original windows only):")
for sp_idx, sp_name in enumerate(SPECIES):
    sp_mask = (y_orig == sp_idx)
    n_sp = sp_mask.sum()
    if n_sp == 0:
        continue
    agree = (pred_orig[sp_mask] == sp_idx).mean()
    bar = "█" * int(agree * 20)
    print(f"    {sp_name:30s}  n={n_sp:4d}  agree={agree:5.1%}  {bar}")

print()
if agreement_all < 1 / N_CLASSES + 0.05:
    print("  ⚠ Agreement near chance — check window timing or download errors above.")
else:
    print("  Agreement is above chance — teacher signal looks plausible.  PASS")

CHECK 5: Teacher–ground-truth agreement on CLEAN (non-augmented) train windows
         (augmented copies have the same teacher vector as their source,
          so we isolate originals to avoid double-counting)

  Original (non-augmented) rows identified: 3146 / 6292
  Overall agreement (teacher argmax == ground truth): 52.77%
  Chance baseline (11 classes):                       9.09%

  Per-species agreement (original windows only):
    American Robin                  n= 290  agree=31.0%  ██████
    Black-capped Chickadee          n= 293  agree=58.7%  ███████████
    Steller's Jay                   n= 259  agree=42.9%  ████████
    Northern Flicker                n= 277  agree=37.5%  ███████
    Song Sparrow                    n= 285  agree=19.6%  ███
    Anna's Hummingbird              n= 279  agree=81.4%  ████████████████
    Dark-eyed Junco                 n= 292  agree=36.6%  ███████
    American Crow                   n= 305  agree=43.9%  ████████
    Pacific Wren              

In [11]:
# ── Value range summary ───────────────────────────────────────────────────────
print("=" * 62)
print("LOGIT VALUE RANGE SUMMARY")
print(f"  min   : {tl.min():.3f}")
print(f"  max   : {tl.max():.3f}")
print(f"  mean  : {tl.mean():.3f}")
print(f"  median: {np.median(tl):.3f}")

# Final gate: if values look wrong, stop before deleting the progress file.
if tl.std() < 1e-6:
    raise ValueError(
        "Saved teacher_logits.npy appears all-zero — something went wrong.\\n"
        "Progress file preserved. Re-run §4 onwards."
    )

# All checks passed — safe to delete the progress file now.
if PROGRESS_F.exists():
    PROGRESS_F.unlink()
    print("Progress file cleaned up.")

print()
print("=" * 62)
print("TEACHER PIPELINE COMPLETE")
print("=" * 62)
print(f"  teacher_logits.npy : {tl.shape}  float32")
print()
print("Next: notebooks/03_train.ipynb — train variants A, B, C.")
print("  Variant C uses teacher_logits.npy for the KD loss.")
print("  Variants A and B ignore it.")

LOGIT VALUE RANGE SUMMARY
  min   : -15.000
  max   : 14.333
  mean  : -8.091
  median: -9.070
Progress file cleaned up.

TEACHER PIPELINE COMPLETE
  teacher_logits.npy : (6292, 11)  float32

Next: notebooks/03_train.ipynb — train variants A, B, C.
  Variant C uses teacher_logits.npy for the KD loss.
  Variants A and B ignore it.
